In [ ]:
# ── Configuration du chemin racine du projet ─────────────────────────────
import os
import sys
parent_dir = os.path.abspath('..')
sys.path.append(parent_dir)  # rend les modules src/ et Config visibles

# ── Installation et import des librairies ────────────────────────────────
!pip install catboost xgboost lightgbm catboost scikit-learn matplotlib pandas wordcloud textblob transformers torch vaderSentiment-fr

import numpy as np
import matplotlib.pyplot as plt

# Chargement des données (S3 ou local selon Config.LOAD_FROM_S3)
import s3fs
import time
import pandas as pd
import logging
logging.basicConfig(level=logging.INFO)  # affiche les logs INFO et supérieurs

# Fonctions de chargement et de preprocessing des données
from src.data_processing.data_load import data_loading
from src.data_processing.data_processing import DataProcessor

In [ ]:
# ── Chargement des données portefeuille (training + validation) ───────────
# On concatène les deux jeux pour avoir une vue complète du portefeuille
df_portefeuille1 = data_loading("training")[0]    # données d'apprentissage (2023-11)
df_portefeuille2 = data_loading("validation")[0]  # données de validation  (2024-11)

df_portefeuille = pd.concat([df_portefeuille1, df_portefeuille2], ignore_index=True)

In [ ]:
# Vérification que les deux fichiers ont les mêmes colonnes avant concaténation
set(df_portefeuille1.columns) == set(df_portefeuille2.columns)

In [ ]:
# Dimensions du portefeuille complet (nb lignes, nb colonnes)
df_portefeuille.shape

In [ ]:
df = df_portefeuille

# ── Extraction des colonnes NPS ───────────────────────────────────────────
# On isole les variables NPS (note, verbatim, date) pour les années N et N-1
# ainsi que la variable cible (résiliation à 6 mois)
colonnes_nps = [
    'client_nps_date_reponse_n',       # date de réponse au sondage NPS (année N)
    'client_nps_note_reco_n',          # note de recommandation 0-10 (année N)
    'client_nps_verbatim_n',           # commentaire libre du client (année N)
    'client_nps_date_reponse_n_moins1',# date de réponse NPS (année N-1)
    'client_nps_note_reco_n_moins1',   # note de recommandation (année N-1)
    'client_nps_verbatim_n_moins1',    # commentaire libre (année N-1)
    'target_resiliation_6mois'         # variable cible : 1 = résilié dans les 6 mois
]

# Extraction dans un sous-DataFrame dédié à l'analyse NPS
df_nps = df[colonnes_nps]

print(df_nps.head())

In [ ]:
# Analyse des valeurs manquantes sur les colonnes NPS
# Les données NPS sont très incomplètes : seule une fraction des clients répond au sondage
df_nps.isna().sum()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from wordcloud import WordCloud
from textblob import TextBlob
from vaderSentiment_fr.vaderSentiment import SentimentIntensityAnalyzer
from transformers import pipeline

# ── Préparation : on garde uniquement les clients ayant répondu aux deux années ──
df_nps = df.dropna(subset=['client_nps_verbatim_n', 'client_nps_verbatim_n_moins1']).copy()

# ── Modèle de sentiment : BERT multilingue (Hugging Face) ────────────────
# Modèle pré-entraîné capable d'analyser des textes en français
# Retourne un label (1 à 5 étoiles) et un score de confiance
sentiment_model = pipeline(
    "sentiment-analysis",
    model="nlptown/bert-base-multilingual-uncased-sentiment"
)

def sentiment_transformer(text):
    """
    Calcule un score de sentiment entre -1 (très négatif) et +1 (très positif).
    Les labels 4 et 5 étoiles → score positif, les autres → score négatif.
    """
    result = sentiment_model(str(text))[0]
    return result["score"] if result["label"] in ["4 stars", "5 stars"] else -result["score"]

def sentiment_label(score):
    """Classe le score de sentiment en 3 catégories : Positif, Neutre, Négatif."""
    if score > 0.1:
        return "Positif"
    elif score < -0.1:
        return "Négatif"
    else:
        return "Neutre"

# ── Application du modèle sur les verbatims N et N-1 ─────────────────────
df_nps["sentiment_score_n"]  = df_nps["client_nps_verbatim_n"].apply(sentiment_transformer)
df_nps["sentiment_score_n1"] = df_nps["client_nps_verbatim_n_moins1"].apply(sentiment_transformer)

df_nps["sentiment_label_n"]  = df_nps["sentiment_score_n"].apply(sentiment_label)
df_nps["sentiment_label_n1"] = df_nps["sentiment_score_n1"].apply(sentiment_label)

# ── Concaténation des verbatims pour les nuages de mots ──────────────────
texte_n  = " ".join(df_nps["client_nps_verbatim_n"].astype(str))
texte_n1 = " ".join(df_nps["client_nps_verbatim_n_moins1"].astype(str))

# Mots vides français à exclure du nuage de mots
stop_words = set([
    'le', 'la', 'les', 'de', 'des', 'du', 'un', 'une', 'et', 'en', 'dans',
    'pour', 'que', 'qui', 'au', 'aux', 'avec', 'ce', 'cette', 'ces', 'est'
])

# ── Génération des nuages de mots ─────────────────────────────────────────
wordcloud_n = WordCloud(width=800, height=400, background_color="white",
                        stopwords=stop_words).generate(texte_n)
wordcloud_n1 = WordCloud(width=800, height=400, background_color="white",
                         stopwords=stop_words).generate(texte_n1)

# ── Statistiques descriptives ─────────────────────────────────────────────
mean_n  = df_nps["client_nps_note_reco_n"].mean()
mean_n1 = df_nps["client_nps_note_reco_n_moins1"].mean()

sent_n  = df_nps["sentiment_label_n"].value_counts().reindex(["Positif", "Neutre", "Négatif"], fill_value=0)
sent_n1 = df_nps["sentiment_label_n1"].value_counts().reindex(["Positif", "Neutre", "Négatif"], fill_value=0)

# ── Visualisation : 4 graphiques en grille 2x2 ───────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Nuage de mots N-1
axes[0, 0].imshow(wordcloud_n1, interpolation="bilinear")
axes[0, 0].axis("off")
axes[0, 0].set_title("WordCloud Verbatims N-1")

# Nuage de mots N
axes[0, 1].imshow(wordcloud_n, interpolation="bilinear")
axes[0, 1].axis("off")
axes[0, 1].set_title("WordCloud Verbatims N")

# Distribution des scores de sentiment (N vs N-1) — en proportion
bins = np.linspace(-1, 1, 10)
axes[1, 0].hist(df_nps["sentiment_score_n1"], bins=bins, alpha=0.6, label="N-1", density=True)
axes[1, 0].hist(df_nps["sentiment_score_n"],  bins=bins, alpha=0.6, label="N",   density=True)
axes[1, 0].set_title("Distribution des scores de sentiment (proportion)")
axes[1, 0].set_xlabel("Score de sentiment")
axes[1, 0].set_ylabel("Proportion")
axes[1, 0].legend()

# Distribution des notes NPS 0-10 (N vs N-1) — en proportion
bins = np.arange(0, 12) - 0.5
counts_n1, _ = np.histogram(df_nps["client_nps_note_reco_n_moins1"], bins=bins)
counts_n, _  = np.histogram(df_nps["client_nps_note_reco_n"],        bins=bins)
prop_n1 = counts_n1 / counts_n1.sum()
prop_n  = counts_n  / counts_n.sum()
x = np.arange(0, 11)
axes[1, 1].bar(x - 0.2, prop_n1, width=0.4, label="N-1")
axes[1, 1].bar(x + 0.2, prop_n,  width=0.4, label="N")
axes[1, 1].set_xticks(x)
axes[1, 1].set_title("Distribution des scores NPS (proportion)")
axes[1, 1].set_xlabel("Score NPS")
axes[1, 1].set_ylabel("Proportion")
axes[1, 1].legend()

plt.tight_layout()
plt.show()

print(f"Note moyenne N-1 : {mean_n1:.2f}")
print(f"Note moyenne N   : {mean_n:.2f}")
print("\nRépartition sentiments N-1 :")
print(sent_n1)
print("\nRépartition sentiments N :")
print(sent_n)

In [ ]:
# ── Taux de résiliation par score NPS ────────────────────────────────────
# On calcule la moyenne de résiliation pour chaque note NPS (0 à 10).
# Cela permet de vérifier si une note basse est bien corrélée à un churn plus élevé.
resiliation_by_nps = df_nps.groupby("client_nps_note_reco_n")["target_resiliation_6mois"].mean()

plt.figure(figsize=(8,5))
resiliation_by_nps.plot(marker="o")

plt.title("Taux de résiliation en fonction du score NPS")
plt.xlabel("Score NPS")
plt.ylabel("Taux de résiliation")
plt.grid()

plt.show()

In [ ]:
import re
import pandas as pd
import matplotlib.pyplot as plt

# ── Détection des mentions de résiliation dans les verbatims ──────────────
# On crée un flag binaire pour identifier les clients qui mentionnent
# explicitement leur intention de partir dans leur commentaire NPS.
# Cette approche NLP simple permet d'extraire un signal d'alerte précoce.

# 1. Liste de mots-clés associés à une intention de résiliation
keywords_resiliation = [
    "résiliation","resiliation","résilier","resilier","résilié","resilie",
    "quitter","partir","changer","annuler","stop","mettre fin","je pars",
    "je vais partir","je pense partir","changer d'assurance","aller ailleurs","RIA","RAD"
]

pattern = "|".join(keywords_resiliation)

# 2. Application du flag sur le verbatim de l'année N
df_nps["flag_resiliation_verbatim"] = (
    df_nps["client_nps_verbatim_n"]
    .str.lower()
    .str.contains(pattern, regex=True, na=False)
)

# 3. Nombre de clients ayant mentionné la résiliation
n_clients = df_nps["flag_resiliation_verbatim"].sum()
print(f"Nombre de clients mentionnant la résiliation : {n_clients}")

# 4. Impact sur le churn : comparaison entre les deux groupes
summary = df_nps.groupby("flag_resiliation_verbatim").agg(
    churn_rate=("target_resiliation_6mois", "mean"),
    count=("target_resiliation_6mois", "size")
)

print("\nRésumé :")
print(summary)

# 5. Visualisation : impact de la mention de résiliation sur le taux de churn
plot_df = summary.copy()
plot_df["churn_rate"] = plot_df["churn_rate"] * 100
plot_df["label"] = ["Pas mentionné", "Mentionné"]

plt.figure(figsize=(6,4))
plt.bar(plot_df["label"], plot_df["churn_rate"])
plt.title("Impact de la mention de résiliation sur le churn")
plt.ylabel("Taux de churn (%)")
plt.tight_layout()
plt.show()

In [ ]:
# ── Identification du segment "Prix négatif" ─────────────────────────────
# On cherche à isoler les clients mécontents du prix :
#   - Ils mentionnent des termes liés au tarif/coût dans leur verbatim NPS
#   - Leur sentiment BERT est négatif (score < 0)
# Ce double critère (lexical + sentiment) réduit les faux positifs.

# Liste de mots-clés liés au prix/tarif
mots_cles_prix = ['prix', 'tarif', 'cher', 'cotisation', 'augmentation', 'coût', 'cout', 'augment']

# Création d'un flag binaire : le client mentionne-t-il le prix ?
pattern = '|'.join(mots_cles_prix)
df_nps['mention_prix'] = df_nps['client_nps_verbatim_n'].str.contains(pattern, case=False, na=False)

# Sous-ensemble : mention du prix ET sentiment négatif → segment "prix négatif"
df_prix_negatif = df_nps[(df_nps['mention_prix'] == True) & (df_nps['sentiment_score_n'] < 0)]

In [ ]:
target = 'target_resiliation_6mois'

# ── Sur-risque du segment "Prix négatif" ──────────────────────────────────
# On compare le taux de churn du segment prix négatif au taux global
# pour quantifier le sur-risque associé à l'insatisfaction tarifaire.

taux_churn_prix   = df_prix_negatif[target].mean() * 100
taux_churn_global = df_nps[target].mean() * 100

print(f"Nombre de clients mécontents du prix : {len(df_prix_negatif)}")
print(f"Taux de churn (Segment Prix) : {taux_churn_prix:.2f}%")
print(f"Taux de churn (Moyenne Globale) : {taux_churn_global:.2f}%")

# L'over-risk mesure combien de fois plus ce segment churne par rapport à la moyenne
sur_risque = taux_churn_prix / taux_churn_global
print(f"Ces clients sont {sur_risque:.1f} fois plus susceptibles de partir.")

In [ ]:
import seaborn as sns

# ── Visualisation : impact de l'insatisfaction tarifaire sur la résiliation ──
# Graphique à barres comparant le taux de churn global vs segment prix négatif.
# La couleur rouge signale le groupe à risque élevé.

data_plot = pd.DataFrame({
    'Groupe': ['Moyenne Globale', 'Mécontents du Prix'],
    'Taux de Churn (%)': [taux_churn_global, taux_churn_prix]
})

plt.figure(figsize=(8, 5))
sns.barplot(x='Groupe', y='Taux de Churn (%)', data=data_plot, palette=['grey', 'red'])
plt.title('Impact de l\'insatisfaction tarifaire sur la résiliation')
plt.ylim(0, max(taux_churn_prix, taux_churn_global) + 5)
plt.show()

In [ ]:
# ── Segmentation NPS standard : Détracteurs / Passifs / Promoteurs ────────
# La méthode NPS segmente les répondants en 3 catégories selon leur note :
#   - Détracteurs  : note 0 à 6  (clients insatisfaits, à risque de churn)
#   - Passifs      : note 7 à 8  (clients neutres)
#   - Promoteurs   : note 9 à 10 (clients très satisfaits, ambassadeurs)

def segment_nps(note):
    if note <= 6:
        return 'Détracteur'
    elif note <= 8:
        return 'Passif'
    else:
        return 'Promoteur'

# Application de la segmentation sur la note NPS de l'année N
df_nps['categorie_nps'] = df_nps['client_nps_note_reco_n'].apply(segment_nps)

# Calcul du taux de churn par segment NPS
target = 'target_resiliation_6mois'
stats_nps = df_nps.groupby('categorie_nps')[target].mean() * 100

print("--- Taux de Churn par Segment NPS ---")
print(stats_nps.sort_values(ascending=False))

In [ ]:
# ── Comparaison NPS structurel vs signal NLP "Prix" ──────────────────────
# Ce graphique superpose deux sources d'information :
#   - Les barres : taux de churn par segment NPS (signal structurel de satisfaction)
#   - La ligne rouge pointillée : taux de churn du segment "prix négatif" (signal NLP)
# Cela permet d'évaluer si le signal NLP apporte une information complémentaire
# au-delà de ce qu'explique déjà la note NPS.

plt.figure(figsize=(10, 6))
sns.barplot(x=stats_nps.index, y=stats_nps.values, palette='coolwarm')
plt.axhline(taux_churn_prix, color='red', linestyle='--', label='Risque "Prix" (NLP)')
plt.title('Taux de Churn par Segment NPS vs Signal NLP "Prix"', fontsize=14)
plt.ylabel('Taux de Churn (%)')
plt.legend()
plt.show()

In [ ]:
# ── Churn du segment "Prix négatif" croisé avec la catégorie NPS ─────────
# On affine l'analyse en croisant l'insatisfaction tarifaire (NLP)
# avec la segmentation NPS (structurelle).
# Résultat : les Détracteurs mécontents du prix sont le sous-groupe le plus à risque.

df_prix_negatif = df_nps[(df_nps['mention_prix'] == True) & (df_nps['sentiment_score_n'] < 0)]

# Taux de churn par catégorie NPS au sein du segment prix négatif
resultat_churn_prix = df_prix_negatif.groupby('categorie_nps')['target_resiliation_6mois'].mean() * 100

print(resultat_churn_prix.sort_values(ascending=False))

In [ ]:
# ── Comparaison graphique : churn global vs churn prix négatif par segment NPS ──
# Ce graphique à barres groupées permet de visualiser l'effet amplificateur
# de l'insatisfaction tarifaire au sein de chaque segment NPS.
# Pour chaque segment, la barre "Prix négatif" est supérieure à la barre "Global",
# ce qui confirme que le signal NLP ajoute de la valeur prédictive.

df_compare = pd.DataFrame({
    'Global': stats_nps,
    'Prix négatif': resultat_churn_prix
})

ax = df_compare.plot(kind='bar', figsize=(8,5))

plt.title("Comparaison du taux de churn par segment NPS")
plt.ylabel("Taux de churn (%)")
plt.xlabel("Segment NPS")
plt.xticks(rotation=0)

# Ajout des valeurs numériques sur chaque barre pour faciliter la lecture
for container in ax.containers:
    ax.bar_label(container, fmt='%.1f', padding=3)

plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
def compute_nps_detail(scores: pd.Series):
    """
    Calcule le score NPS et la répartition des segments à partir d'une série de notes.

    Formule NPS = % Promoteurs - % Détracteurs
    - Promoteurs  : note 9 ou 10
    - Passifs     : note 7 ou 8
    - Détracteurs : note 0 à 6

    Paramètre
    ---------
    scores : pd.Series
        Série de notes NPS (entiers entre 0 et 10, valeurs manquantes ignorées).

    Retourne
    --------
    dict avec les clés : 'NPS', '% promoteurs', '% passifs', '% détracteurs'
    """
    scores = scores.dropna()

    pct_promoteurs   = ((scores >= 9) & (scores <= 10)).mean() * 100
    pct_passifs      = ((scores >= 7) & (scores <= 8)).mean() * 100
    pct_detracteurs  = ((scores >= 0) & (scores <= 6)).mean() * 100

    nps = pct_promoteurs - pct_detracteurs

    return {
        "NPS": nps,
        "% promoteurs": pct_promoteurs,
        "% passifs": pct_passifs,
        "% détracteurs": pct_detracteurs
    }

In [ ]:
# Calcul du NPS détaillé pour l'année N (portefeuille complet)
scores = df["client_nps_note_reco_n"].dropna()
compute_nps_detail(scores)

In [ ]:
# Calcul du NPS détaillé pour l'année N-1 (portefeuille complet)
# Permet de comparer l'évolution du score NPS d'une année sur l'autre.
scores = df["client_nps_note_reco_n_moins1"].dropna()
compute_nps_detail(scores)

In [ ]:
# Vérification rapide du score NPS N-1 avec la formule directe
# NPS = (% Promoteurs - % Détracteurs) × 100
scores = df["client_nps_note_reco_n_moins1"].dropna()
nps = ((scores >= 9).mean() - (scores <= 6).mean()) * 100
print(f"NPS = {nps:.2f}")

In [ ]:
def nps_summary(scores: pd.Series) -> pd.DataFrame:
    """
    Calcule un tableau récapitulatif du NPS à partir d'une série de notes.

    Retourne un DataFrame avec 4 colonnes :
      - Promoteurs (%)  : part des notes 9-10
      - Passifs (%)     : part des notes 7-8
      - Détracteurs (%) : part des notes 0-6
      - NPS             : score net = Promoteurs - Détracteurs

    Paramètre
    ---------
    scores : pd.Series
        Série de notes NPS (entiers entre 0 et 10).

    Retourne
    --------
    pd.DataFrame à une ligne, arrondi à 2 décimales.
    """
    scores = scores.dropna()
    total  = len(scores)

    data = {
        "Promoteurs (%)":   (scores >= 9).mean() * 100,
        "Passifs (%)":      ((scores >= 7) & (scores <= 8)).mean() * 100,
        "Détracteurs (%)":  (scores <= 6).mean() * 100,
    }

    data["NPS"] = data["Promoteurs (%)"] - data["Détracteurs (%)"]

    return pd.DataFrame([data]).round(2)

In [ ]:
# Tableau NPS synthétique pour l'année N
df_nps = nps_summary(df["client_nps_note_reco_n"])
print(df_nps)

In [ ]:
# Tableau NPS synthétique pour l'année N-1
# Le NPS passe de 18.89 (N-1) à 20.15 (N) : légère amélioration de la satisfaction.
df_nps = nps_summary(df["client_nps_note_reco_n_moins1"])
print(df_nps)

In [ ]:
# ── NPS par groupe de churn ───────────────────────────────────────────────
# On calcule le NPS séparément pour les clients qui ont résilié (churn=1)
# et ceux qui sont restés (churn=0).
# Résultat attendu : le NPS des churners est significativement inférieur,
# ce qui valide la pertinence du NPS comme indicateur de rétention.
df.groupby("target_resiliation_6mois")["client_nps_note_reco_n"].apply(nps_summary)

In [ ]:
# ── Installation de BERTopic et sentence-transformers ────────────────────
# BERTopic est un modèle de topic modeling basé sur des embeddings BERT.
# sentence-transformers fournit le modèle d'embeddings multilingue utilisé
# pour encoder les verbatims avant la modélisation des topics.
pip install bertopic sentence-transformers

In [ ]:
import re
import pandas as pd
import matplotlib.pyplot as plt
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer

# ── Pré-traitement du texte pour BERTopic (année N) ──────────────────────
# On nettoie les verbatims avant de les passer dans le modèle :
#   - Mise en minuscules et suppression de la ponctuation
#   - Suppression des mots vides français (stop words)
#   - Exclusion des mots trop courts (≤ 2 caractères)

stop_words_fr = {
    "le","la","les","de","des","du","un","une","et","en","dans","pour","que","qui",
    "au","aux","avec","ce","cette","ces","est","je","j","ai","ne","plus","sur",
    "par","mon","ma","mes","vos","vous","nous","il","elle","on","a","avoir","être",
    "très","car","donc","depuis","comme","fait","faites","faire","été","suis",
    "sont","plusieurs","tout","tous","toute","toutes","aucun","aucune","chez","après",
    "avant","alors","aussi","mais","ou","or","ni"
}

def clean_text(text):
    """Nettoie un verbatim : minuscules, sans ponctuation, sans stop words."""
    text  = str(text).lower()
    text  = re.sub(r"[^\w\s]", " ", text)
    words = [w for w in text.split() if w not in stop_words_fr and len(w) > 2]
    return " ".join(words)

# Filtre : clients ayant un verbatim N et une cible disponibles
df_topic = df.dropna(subset=["client_nps_verbatim_n", "target_resiliation_6mois"]).copy()
df_topic["text_clean"] = df_topic["client_nps_verbatim_n"].apply(clean_text)

# ── Modélisation des topics sur les verbatims N ───────────────────────────
# On utilise le modèle multilingue MiniLM pour générer des embeddings,
# puis BERTopic regroupe automatiquement les verbatims par sujet.
# Le topic -1 correspond aux "outliers" (verbatims non assignés à un topic).

embedding_model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

topic_model = BERTopic(
    embedding_model=embedding_model,
    language="multilingual",
    verbose=False
)

topics, probs = topic_model.fit_transform(df_topic["text_clean"].tolist())
df_topic["topic"] = topics

# Récupération des labels de topics (3 mots représentatifs par topic)
topic_info = topic_model.get_topic_info().copy()
topic_info = topic_info[topic_info["Topic"] != -1].reset_index(drop=True)

topic_labels = {}
for topic_id in topic_info["Topic"]:
    words = [w for w, _ in topic_model.get_topic(topic_id)[:3]]
    topic_labels[topic_id] = " / ".join(words)

df_topic["topic_label"] = df_topic["topic"].map(topic_labels).fillna("Outliers / Autres")

# ── Agrégation : taux de churn et taille par topic ────────────────────────
topic_summary = (
    df_topic[df_topic["topic"] != -1]
    .groupby(["topic", "topic_label"])
    .agg(
        churn_rate=("target_resiliation_6mois", "mean"),
        n_clients=("target_resiliation_6mois", "size")
    )
    .reset_index()
)

In [ ]:
# ── Modélisation BERTopic sur les verbatims N-1 ───────────────────────────
# Même pipeline que pour l'année N, appliqué aux verbatims de l'année précédente.
# Cela permet de comparer les thèmes émergents d'une année à l'autre
# et d'identifier des signaux de mécontentement récurrents ou nouveaux.

df_topic_moins1 = df.dropna(subset=["client_nps_verbatim_n_moins1", "target_resiliation_6mois"]).copy()
df_topic_moins1["text_clean"] = df_topic_moins1["client_nps_verbatim_n_moins1"].apply(clean_text)

embedding_model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

topic_model = BERTopic(
    embedding_model=embedding_model,
    language="multilingual",
    verbose=False
)

topics, probs = topic_model.fit_transform(df_topic_moins1["text_clean"].tolist())
df_topic_moins1["topic"] = topics

# Labels des topics N-1 (3 mots représentatifs)
topic_info = topic_model.get_topic_info().copy()
topic_info = topic_info[topic_info["Topic"] != -1].reset_index(drop=True)

topic_labels = {}
for topic_id in topic_info["Topic"]:
    words = [w for w, _ in topic_model.get_topic(topic_id)[:3]]
    topic_labels[topic_id] = " / ".join(words)

df_topic_moins1["topic_label"] = df_topic_moins1["topic"].map(topic_labels).fillna("Outliers / Autres")

# Agrégation : churn et volume par topic N-1
topic_summary_moins1 = (
    df_topic_moins1[df_topic_moins1["topic"] != -1]
    .groupby(["topic", "topic_label"])
    .agg(
        churn_rate=("target_resiliation_6mois", "mean"),
        n_clients=("target_resiliation_6mois", "size")
    )
    .reset_index()
)

In [ ]:
# Aperçu du tableau des topics N : chaque ligne = un topic identifié
# avec son label (3 mots clés), son taux de churn moyen et le nombre de clients.
topic_summary

In [ ]:
# Aperçu du tableau des topics N-1 : même structure que topic_summary,
# mais construit sur les verbatims de l'année précédente.
topic_summary_moins1

In [ ]:
# ── Top topics par groupe de churn — Année N ─────────────────────────────
# On calcule la proportion de chaque topic au sein des churners (churn=1)
# et des non-churners (churn=0), puis on affiche les 10 topics les plus fréquents.
# Cela révèle quels sujets sont sur-représentés chez les clients qui partent.

top_topics_by_churn = (
    df_topic[df_topic["topic"] != -1]
    .groupby(["target_resiliation_6mois", "topic_label"])
    .size()
    .reset_index(name="count")
)

# Calcul des proportions au sein de chaque groupe de churn (somme = 1 par groupe)
top_topics_by_churn["proportion"] = (
    top_topics_by_churn.groupby("target_resiliation_6mois")["count"]
    .transform(lambda x: x / x.sum())
)

top_topics_by_churn = top_topics_by_churn.sort_values(
    ["target_resiliation_6mois", "count"], ascending=[True, False]
)

# Visualisation côte à côte : Non-churners (gauche) vs Churners (droite)
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharex=True)

for i, churn_value in enumerate([0, 1]):
    plot_df = (
        top_topics_by_churn[top_topics_by_churn["target_resiliation_6mois"] == churn_value]
        .head(10)
        .sort_values("proportion", ascending=True)
    )

    axes[i].barh(plot_df["topic_label"], plot_df["proportion"])
    axes[i].set_title(f"Top topics - Churn = {churn_value}")
    axes[i].set_xlabel("Proportion")
    axes[i].set_ylabel("")

plt.tight_layout()
plt.show()

In [ ]:
# ── Top topics par groupe de churn — Année N-1 ───────────────────────────
# Même analyse que pour l'année N, mais appliquée aux verbatims N-1.
# Permet de détecter des signaux précoces : les sujets de mécontentement
# en N-1 chez les clients qui ont finalement churné en N.

top_topics_by_churn_moins1 = (
    df_topic_moins1[df_topic_moins1["topic"] != -1]
    .groupby(["target_resiliation_6mois", "topic_label"])
    .size()
    .reset_index(name="count")
)

top_topics_by_churn_moins1["proportion"] = (
    top_topics_by_churn_moins1.groupby("target_resiliation_6mois")["count"]
    .transform(lambda x: x / x.sum())
)

top_topics_by_churn_moins1 = top_topics_by_churn_moins1.sort_values(
    ["target_resiliation_6mois", "count"], ascending=[True, False]
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharex=True)

for i, churn_value in enumerate([0, 1]):
    plot_df = (
        top_topics_by_churn_moins1[top_topics_by_churn_moins1["target_resiliation_6mois"] == churn_value]
        .head(10)
        .sort_values("proportion", ascending=True)
    )

    axes[i].barh(plot_df["topic_label"], plot_df["proportion"])
    axes[i].set_title(f" (N-1)Top topics - Churn = {churn_value}")
    axes[i].set_xlabel("Proportion")
    axes[i].set_ylabel("")

plt.tight_layout()
plt.show()

In [ ]:
# ── Top topics par segment NPS — Année N ─────────────────────────────────
# On classifie les clients en 3 groupes NPS (Détracteurs/Passifs/Promoteurs)
# puis on identifie les 5 sujets les plus fréquents dans chaque groupe.
# Cela révèle les thèmes distinctifs de satisfaction et d'insatisfaction.

def nps_group(score):
    """Classifie une note NPS en segment : Détracteurs, Passifs, Promoteurs."""
    if score <= 6:
        return "Détracteurs"
    elif score <= 8:
        return "Passifs"
    else:
        return "Promoteurs"

df_topic["nps_group"] = df_topic["client_nps_note_reco_n"].apply(nps_group)

# Comptage des topics par segment NPS
top_topics_by_nps = (
    df_topic[df_topic["topic"] != -1]
    .groupby(["nps_group", "topic_label"])
    .size()
    .reset_index(name="count")
)

# Proportions au sein de chaque segment (somme = 1 par groupe NPS)
top_topics_by_nps["proportion"] = (
    top_topics_by_nps.groupby("nps_group")["count"]
    .transform(lambda x: x / x.sum())
)

top_topics_by_nps = top_topics_by_nps.sort_values(
    ["nps_group", "count"], ascending=[True, False]
)

# Top 5 topics par segment NPS
top_5_topics_by_nps = (
    top_topics_by_nps.groupby("nps_group")
    .head(5)
    .reset_index(drop=True)
)

print(top_5_topics_by_nps)

In [ ]:
# ── Top topics par segment NPS — Année N-1 ───────────────────────────────
# Même analyse que pour l'année N, appliquée aux verbatims N-1.
# On peut comparer les thèmes dominants d'une année à l'autre pour chaque segment.

df_topic_moins1["nps_group"] = df_topic_moins1["client_nps_note_reco_n"].apply(nps_group)

top_topics_by_nps_moins1 = (
    df_topic_moins1[df_topic_moins1["topic"] != -1]
    .groupby(["nps_group", "topic_label"])
    .size()
    .reset_index(name="count")
)

top_topics_by_nps_moins1["proportion"] = (
    top_topics_by_nps_moins1.groupby("nps_group")["count"]
    .transform(lambda x: x / x.sum())
)

top_topics_by_nps_moins1 = top_topics_by_nps_moins1.sort_values(
    ["nps_group", "count"], ascending=[True, False]
)

# Top 5 topics par segment NPS pour N-1
top_5_topics_by_nps_moins1 = (
    top_topics_by_nps_moins1.groupby("nps_group")
    .head(5)
    .reset_index(drop=True)
)

print(top_5_topics_by_nps_moins1)

In [ ]:
# ── Visualisation : top 10 topics par segment NPS — Année N ──────────────
# Graphiques à barres horizontales pour les 3 segments NPS.
# Les Détracteurs tendent à mentionner des sujets négatifs (délais, tarifs),
# les Promoteurs des sujets positifs (écoute, rapidité des remboursements).

groups = ["Détracteurs", "Passifs", "Promoteurs"]

fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharex=True)

for i, group in enumerate(groups):
    plot_df = (
        top_topics_by_nps[top_topics_by_nps["nps_group"] == group]
        .head(10)
        .sort_values("proportion", ascending=True)
    )

    axes[i].barh(plot_df["topic_label"], plot_df["proportion"])
    axes[i].set_title(f"Top topics - {group}")
    axes[i].set_xlabel("Proportion")

plt.tight_layout()
plt.show()

In [ ]:
# ── Visualisation : top 10 topics par segment NPS — Année N-1 ────────────
# Même graphique que pour l'année N, construit sur les verbatims N-1.
# La comparaison des deux années permet de suivre l'évolution des préoccupations
# des clients selon leur niveau de satisfaction.

groups = ["Détracteurs", "Passifs", "Promoteurs"]

fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharex=True)

for i, group in enumerate(groups):
    plot_df = (
        top_topics_by_nps_moins1[top_topics_by_nps_moins1["nps_group"] == group]
        .head(10)
        .sort_values("proportion", ascending=True)
    )

    axes[i].barh(plot_df["topic_label"], plot_df["proportion"])
    axes[i].set_title(f"(N-1)Top topics - {group}")
    axes[i].set_xlabel("Proportion")

plt.tight_layout()
plt.show()

In [ ]:
# Tableau NPS synthétique N-1 (résultat de la dernière exécution de nps_summary)
# Note : df_nps ici contient le résumé NPS et non les données clients individuelles.
df_nps

In [ ]:
# ── Preprocessing du portefeuille complet avec DataProcessor ─────────────
# On applique le pipeline de transformation standard du projet Alptis
# (encodage ordinal, target encoding, normalisation) sur les données training
# et validation, en évitant toute fuite de données (data leakage).
#
# Principe :
#   1. On entraîne le DataProcessor sur le jeu d'entraînement (fit + transform)
#   2. On transfère les objets ajustés (encodeurs, scaler) au DataProcessor de test
#   3. On applique uniquement une transformation (transform) sur le jeu de test
#
# Ainsi, le scaler et les encodeurs sont toujours ajustés sur les données
# d'entraînement uniquement, jamais sur les données de test.

##1. Pre-process training data (scaler and encoder are saved during this process)
train_processor = DataProcessor(mode="training")
df_train = train_processor.run()  # fit + transform sur le jeu d'entraînement

##2. Pre-process test data (by using the same scaler/encoder fitted on training dataset)
test_processor = DataProcessor(mode="validation")

# Transfert des objets ajustés : évite le data leakage sur target encoding et normalisation
test_processor.target_encoding_maps = train_processor.target_encoding_maps
test_processor.ordinal_maps          = train_processor.ordinal_maps
test_processor.encoded_columns       = train_processor.encoded_columns
test_processor.scaler                = train_processor.scaler
test_processor.normalized_columns    = train_processor.normalized_columns

df_test = test_processor.run_transform()  # transform uniquement sur le jeu de test

In [ ]:
# Aperçu du portefeuille complet (training + validation concaténés)
# 171 025 clients × 47 variables avant preprocessing
df_portefeuille

In [ ]:
# Liste de toutes les colonnes disponibles dans le portefeuille brut
# Utile pour identifier les variables NPS, cotisations et cible disponibles
df_portefeuille.columns

In [ ]:
# ── Analyse des valeurs manquantes du portefeuille ───────────────────────
# Les données NPS sont très lacunaires : ~94% des clients n'ont pas répondu
# au sondage NPS, ce qui limite les analyses NLP à un sous-ensemble restreint.
# Les cotisations N-1 et N-2 ont aussi beaucoup de NaN (clients récents).
nan_stats = pd.DataFrame({
    'nb_nan': df_portefeuille.isnull().sum(),
    'pct_nan': (df_portefeuille.isnull().sum() / len(df_portefeuille) * 100).round(2)
})
nan_stats = nan_stats[nan_stats['nb_nan'] > 0].sort_values('pct_nan', ascending=False)
print(nan_stats.to_string())

In [ ]:
# ── Estimation du taux de churn réel N-1 ─────────────────────────────────
# On estime le churn de l'année N-1 à partir des cotisations annualisées :
#
#   - Nouveaux clients en N-1  : ont une cotisation N mais pas de cotisation N-1
#     (ils n'existaient pas encore dans le portefeuille l'année d'avant)
#   - Clients fidèles          : ont une cotisation N ET une cotisation N-1
#   - Churners estimés N-1     : clients présents en N-1 mais absents en N
#     → calculés par différence : total N-1 - fidèles - nouveaux
#
# NB : cette estimation est approximative car certains NaN peuvent résulter
#      d'une absence de contrat actif plutôt que d'un vrai churn.

nouveaux_N1 = (
    df_portefeuille['client_cotisations_annualisees_n_moins1'].isna() & 
    df_portefeuille['client_cotisations_annualisees_n'].notna()
).sum()

enciens_N1 = (
    df_portefeuille['client_cotisations_annualisees_n_moins1'].notna() & 
    df_portefeuille['client_cotisations_annualisees_n'].notna()
).sum()

clients_N1 = df_portefeuille['client_cotisations_annualisees_n'].notna().sum()

churners_n1 = clients_N1 - enciens_N1 - nouveaux_N1

print(f"Nouveaux en N-1      : {nouveaux_N1}")
print(f"Churners estimés N-1 : {churners_n1}")
print(f"Taux de churn N-1    : {churners_n1 / 94869 * 100:.1f}%")

In [ ]:
# Nombre de clients "anciens" : présents en N-1 ET en N (fidèles sur la période)
enciens_N1

In [ ]:
# Nombre de nouveaux clients en N (absents en N-1, apparus en N)
nouveaux_N1

In [ ]:
# ── Statistiques globales de churn sur le portefeuille complet ───────────
# Le taux de résiliation à 6 mois est calculé sur l'ensemble du portefeuille
# (training + validation). Il constitue la référence pour toutes les comparaisons
# de sur-risque effectuées dans ce notebook.
taux = df_portefeuille["target_resiliation_6mois"].mean() * 100
print(f"Taux de résiliation (6 mois) : {taux:.1f}%")
print(f"Nb résiliations              : {df_portefeuille['target_resiliation_6mois'].sum()}")
print(f"Nb total clients             : {len(df_portefeuille)}")